In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Step 2: Install required packages
!pip install statsmodels tqdm joblib

In [ ]:
# Step 3: Import libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm
from joblib import Parallel, delayed
from tqdm.notebook import tqdm  # Progress bar for Colab
import os

# Step 4: Navigate to the folder
folder_path = '/content/drive/MyDrive/SW_IFL'
os.chdir(folder_path)
!ls  # Verify CSV file is listed

Revised_National_TemperaturePattern.csv  Untitled0.ipynb


In [ ]:
# Step 5: Load data
file_name = 'Revised_National_TemperaturePattern.csv'
df = pd.read_csv(file_name)

# Preview data
print(f"Loaded {len(df)} census tracts")
df.head()

Loaded 33000 census tracts


,GISJOIN,GEOID10,GEOID,Tair_Max_2020,Tair_Min_2020,Tair_Max_2019,Tair_Min_2019,Tair_Max_2018,Tair_Min_2018,Tair_Max_2017,...,Tair_Max_1984,Tair_Min_1984,Tair_Max_1983,Tair_Min_1983,Tair_Max_1982,Tair_Min_1982,Tair_Max_1981,Tair_Min_1981,Tair_Max_1980,Tair_Min_1980
0,G0100090050200,1009050200,1009050200,30.754328,20.270399,31.376158,20.051414,31.534622,20.187006,30.097128,...,31.510578,18.431915,32.391903,18.652624,31.252798,18.942026,32.400208,19.953487,34.004864,19.644571
1,G0100090050400,1009050400,1009050400,30.173449,19.871601,30.782618,19.642181,30.985058,19.750694,29.696087,...,31.158432,18.178434,32.154678,18.249399,30.933069,18.495979,32.050064,19.464449,33.743046,19.335335
2,G0100090050601,1009050601,1009050601,31.135841,20.279201,31.763857,20.063005,31.876072,20.425669,30.386658,...,31.608648,18.331493,32.457371,18.634937,31.293322,18.914326,32.638756,19.837034,34.150230,19.745728
3,G0100730000100,1073000100,1073000100,31.253239,20.654032,32.051422,20.445250,32.045097,20.821358,30.625608,...,31.420153,18.652348,32.301022,19.223566,31.346663,19.367521,32.569282,20.347088,34.003868,20.107130
4,G0100730000300,1073000300,1073000300,31.452173,20.772066,32.239784,20.573914,32.154347,20.959131,30.723913,...,31.507391,18.685761,32.360542,19.298479,31.416739,19.423479,32.665108,20.421196,34.072933,20.171196


In [ ]:
# Step 6: Define the trend calculation function
def calculate_trends(row):
    """Calculate linear trends for max and min temperatures."""
    # Extract all temperature columns
    max_cols = [col for col in row.index if 'Tair_Max' in col]
    min_cols = [col for col in row.index if 'Tair_Min' in col]

    # Get years from column names
    years = np.array([int(col.split('_')[-1]) for col in max_cols])

    # Get temperature values
    max_temps = row[max_cols].values.astype(float)
    min_temps = row[min_cols].values.astype(float)

    # Function to fit model
    def fit_model(y, years):
        X = sm.add_constant(years)  # Add intercept
        model = sm.OLS(y, X, missing='drop').fit()
        return {
            'slope': model.params[1],  # °C/year change
            'p_value': model.pvalues[1],
            'r_squared': model.rsquared,
            'std_error': model.bse[1]
        }

    # Calculate trends
    max_results = fit_model(max_temps, years)
    min_results = fit_model(min_temps, years)

    return {
        'GISJOIN': row['GISJOIN'],
        'GEOID10': row['GEOID10'],
        'GEOID': row['GEOID'],
        'max_slope': max_results['slope'],
        'max_p_value': max_results['p_value'],
        'max_r_squared': max_results['r_squared'],
        'min_slope': min_results['slope'],
        'min_p_value': min_results['p_value'],
        'min_r_squared': min_results['r_squared'],
        'n_years': len(years)
    }

In [ ]:
# Step 7: Process all tracts (with progress bar)
print("Calculating trends for all tracts...")
results = Parallel(n_jobs=-1)(
    delayed(calculate_trends)(row)
    for _, row in tqdm(df.iterrows(), total=len(df))
)

# Convert to DataFrame
results_df = pd.DataFrame(results)

Calculating trends for all tracts...


  0%|          | 0/33000 [00:00<?, ?it/s]

In [ ]:
# Step 8: Save results back to Google Drive
output_file = 'Temperature_Trends_Results.csv'
results_df.to_csv(output_file, index=False)

print(f"Analysis complete! Results saved to {output_file}")
print("Summary statistics:")
print(results_df[['max_slope', 'min_slope']].describe())

Analysis complete! Results saved to Temperature_Trends_Results.csv
Summary statistics:
          max_slope     min_slope
count  33000.000000  33000.000000
mean       0.008138      0.037886
std        0.018729      0.016186
min       -0.056484     -0.053552
25%       -0.005231      0.027780
50%        0.007395      0.037411
75%        0.020978      0.046000
max        0.129897      0.131768
